<a href="https://colab.research.google.com/github/Killua-002/FlyRankIntrnship/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup — connect to the warehouse

This notebook runs against the real ~79M-row warehouse release, not the starter CSV. Request access once at the dataset page, create a **READ** token, then paste it via the prompt below (never hardcode it — this repo is public).

In [1]:
%pip -q install duckdb

import os, getpass, duckdb, pandas as pd, numpy as np

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# Mid-panel month for all iteration below — NOT the sealed final month (2026-06).
MONTH_START = "2026-03-01"
MONTH_END   = "2026-03-31"
print("Connected. Working month:", MONTH_START, "to", MONTH_END)

Paste your Hugging Face READ token (hf_...): ··········
Connected. Working month: 2026-03-01 to 2026-03-31


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Contract, in plain words:**

1. **One row** = one content item's aggregated search performance for a single calendar month (client × content × month grain), split into first-half (days 1–15) and second-half (days 16–end) windows within that month.
2. **Table(s):** `fact_content_daily_performance` (daily grain, aggregated up to month-half here), joined to `dim_content` for static fields like `word_count`-equivalent and `days_since_last_update`, and `dim_clients` only to confirm coverage — not joined into the model frame.
3. **Time window:** March 2026 (`2026-03-01` to `2026-03-31`) — a mid-panel month, per the assignment's own warning that the final month (`2026-06`) is a sealed test month and must never be used to develop label logic.
4. **What I'd predict:** a proxy label, `declining_h2`, defined as second-half impressions dropping more than 20% versus first-half impressions, for content items with enough first-half volume to trust the comparison. This is the same 'proxy, not observed future outcome' pattern flagged in w02 — a within-month comparison, not a true forward-looking label.
5. **One thing I deliberately exclude:** `gsc_avg_position` change between halves. It's highly correlated with the impressions swing I'm using for the label (position and impressions move together mechanically), so including it as a feature risks a softer version of the same leakage the trap in part 3 demonstrates on purpose — better to leave it out here and treat it as a candidate to test carefully later, not fold in by default.

In [2]:
# Verify the contract statement above: does the warehouse actually cover this window,
# and does fact_daily have the grain and columns I just claimed?
con.sql(f"""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS min_d, MAX(report_date) AS max_d
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '{MONTH_START}' AND '{MONTH_END}'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,min_d,max_d
0,9841378,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Field | Bucket | Why |
|---|---|---|
| `imp_h1`, `clk_h1`, `pos_h1`, `ctr_h1` | Feature | All computed only from days 1–15 — knowable at the day-15 decision point. |
| `days_since_last_update` (from `dim_content`) | Feature | Static/slow-changing field, known at any point, not derived from the outcome window. |
| `imp_h2` | Label input | Second-half impressions — used only to build `declining_h2`, never as a feature (this is the field the trap in Section 3 deliberately misuses). |
| `declining_h2` | Label / proxy | The target: did impressions drop >20% from h1 to h2. |
| `client_hash_id`, `content_hash_id` | Context | Grouping and holdout-split keys only — pseudonyms, never features (same rule as the starter CSV). |
| `access_profile` (from `dim_clients`) | Excluded | Internal FlyRank product/contract metadata about the client relationship, not a signal about the content itself — out of scope for this lane and a plausible route to indirectly identifying a client. |
| `gsc_avg_position` (h2 or full-month) | Excluded | Moves mechanically with the impressions swing that defines the label — see the exclusion reasoning in Section 1. |

In [3]:
# No new query needed here — this section's claims are verified by the grain/count/availability
# queries in Section 3, which run against these exact fields.
fields_table = pd.DataFrame([
    ("imp_h1, clk_h1, pos_h1, ctr_h1", "feature"),
    ("days_since_last_update",          "feature"),
    ("imp_h2",                          "label input"),
    ("declining_h2",                    "label/proxy"),
    ("client_hash_id, content_hash_id", "context"),
    ("access_profile",                  "excluded"),
    ("gsc_avg_position (h2)",           "excluded"),
], columns=["field", "bucket"])
fields_table

,field,bucket
0,"imp_h1, clk_h1, pos_h1, ctr_h1",feature
1,days_since_last_update,feature
2,imp_h2,label input
3,declining_h2,label/proxy
4,"client_hash_id, content_hash_id",context
5,access_profile,excluded
6,gsc_avg_position (h2),excluded


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### 3a. Grain check
*Claim: one row = one (client, content) pair for this month-half grain. Verify: group by that key and confirm nothing groups to more than one row.*

In [4]:
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '{MONTH_START}' AND '{MONTH_END}'
    GROUP BY 1, 2
    HAVING COUNT(*) > 31
    LIMIT 5
""").df()
print(f"Rows violating the grain (more than 31 daily rows in a 31-day month): {len(grain_check)}")
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows violating the grain (more than 31 daily rows in a 31-day month): 0


,client_hash_id,content_hash_id,n


### 3b. Row count + date span
*Claim: March 2026 is fully present in the panel, at daily grain, for a meaningful slice of content items.*

In [5]:
counts = con.sql(f"""
    SELECT COUNT(*) AS row_count,
           COUNT(DISTINCT client_hash_id)  AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content,
           MIN(report_date) AS min_d, MAX(report_date) AS max_d
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '{MONTH_START}' AND '{MONTH_END}'
""").df()
counts

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,n_clients,n_content,min_d,max_d
0,9841378,55,331437,2026-03-01,2026-03-31


### 3c. Availability
*Claim: not every client has usable GA4/GSC data this early or this late in their history. Filter with `IS TRUE` and show the survival rate.*

In [6]:
availability = con.sql(f"""
    WITH month_rows AS (
        SELECT f.*, d.gsc_data_start
        FROM {TABLES['fact_daily']} f
        JOIN {TABLES['dim_clients']} d USING (client_hash_id)
        WHERE f.report_date BETWEEN '{MONTH_START}' AND '{MONTH_END}'
    )
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM month_rows
""").df()
availability["ga4_available_pct"] = (availability["ga4_available_rows"] / availability["total_rows"] * 100).round(1)
availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows,ga4_available_pct
0,9841378,413966.0,4.2


### 3d. Five features (max), each knowable at the decision moment
*Decision moment = day 15 of the month. Every feature below uses only days 1–15; the label uses days 16–end.*

In [7]:
feature_frame = con.sql(f"""
    SELECT
        f.client_hash_id, f.content_hash_id,
        SUM(CASE WHEN f.report_date <= DATE '{MONTH_START}' + INTERVAL 14 DAY
                 THEN f.gsc_impressions ELSE 0 END) AS imp_h1,
        SUM(CASE WHEN f.report_date <= DATE '{MONTH_START}' + INTERVAL 14 DAY
                 THEN f.gsc_clicks ELSE 0 END)      AS clk_h1,
        AVG(CASE WHEN f.report_date <= DATE '{MONTH_START}' + INTERVAL 14 DAY
                 THEN f.gsc_avg_position END)        AS pos_h1,
        SUM(CASE WHEN f.report_date >  DATE '{MONTH_START}' + INTERVAL 14 DAY
                 THEN f.gsc_impressions ELSE 0 END) AS imp_h2
    FROM {TABLES['fact_daily']} f
    WHERE f.report_date BETWEEN '{MONTH_START}' AND '{MONTH_END}'
    GROUP BY 1, 2
    HAVING imp_h1 >= 100   -- availability filter: need real first-half volume to trust the comparison
""").df()

# dim_content has no ready-made 'days_since_last_update' column — it has content_updated_date
# instead, so I derive the same concept: days between the last update and my day-15 decision point.
content_static = con.sql(f"""
    SELECT content_hash_id,
           DATE_DIFF('day', content_updated_date, DATE '{MONTH_START}' + INTERVAL 14 DAY)
               AS days_since_last_update
    FROM {TABLES['dim_content']}
""").df()

feature_frame = feature_frame.merge(content_static, on="content_hash_id", how="left")
feature_frame["ctr_h1"] = feature_frame["clk_h1"] / feature_frame["imp_h1"]

print(f"{len(feature_frame):,} content items with enough first-half volume this month\n")
print("Features, and why each is knowable at the day-15 decision point:")
print("  imp_h1                  — already observed by day 15, nothing forward-looking")
print("  clk_h1                  — same window as imp_h1, both closed by day 15")
print("  pos_h1                  — average of daily positions already observed by day 15")
print("  ctr_h1                  — derived purely from clk_h1/imp_h1, both closed by day 15")
print("  days_since_last_update  — derived from content_updated_date, known regardless of which day you ask")
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

77,540 content items with enough first-half volume this month

Features, and why each is knowable at the day-15 decision point:
  imp_h1                  — already observed by day 15, nothing forward-looking
  clk_h1                  — same window as imp_h1, both closed by day 15
  pos_h1                  — average of daily positions already observed by day 15
  ctr_h1                  — derived purely from clk_h1/imp_h1, both closed by day 15
  days_since_last_update  — derived from content_updated_date, known regardless of which day you ask


,client_hash_id,content_hash_id,imp_h1,clk_h1,pos_h1,imp_h2,days_since_last_update,ctr_h1
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,4173.0,6.0,6.327311,2350.0,-113,0.001438
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,245.0,0.0,3.906852,208.0,-64,0.000000
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,3705.0,3.0,6.473735,1925.0,-66,0.000810
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,2440.0,8.0,7.259861,2504.0,-113,0.003279
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,240.0,1.0,3.860842,189.0,-64,0.004167


### 3e. The trap — add a label-derived column on purpose
*Build `declining_h2` from `imp_h2`, then add `imp_h2` itself in as a "feature." Watch the score jump toward perfect. Then delete it and keep the honest number.*

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

model_data = feature_frame.dropna(subset=["pos_h1", "days_since_last_update"]).copy()
model_data["declining_h2"] = (model_data["imp_h2"] < 0.8 * model_data["imp_h1"]).astype(int)

honest_feats = ["imp_h1", "clk_h1", "pos_h1", "ctr_h1", "days_since_last_update"]
leaky_feats  = honest_feats + ["imp_h2"]  # <-- the trap: this is literally in the label formula

y = model_data["declining_h2"]

def quick_score(feats):
    X = model_data[feats]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    clf = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    return accuracy_score(y_te, clf.predict(X_te))

leaky_score  = quick_score(leaky_feats)
honest_score = quick_score(honest_feats)

print(f"WITH imp_h2 (leaky):    accuracy = {leaky_score:.3f}  <- looks amazing, and it should worry you")
print(f"WITHOUT imp_h2 (honest): accuracy = {honest_score:.3f}  <- this is the real number")
print("\nimp_h2 is in the label formula itself (declining_h2 = imp_h2 < 0.8*imp_h1),")
print("so handing it to the model as a feature is handing it the answer. Deleting it and keeping")
print("the honest score is the actual deliverable here — not the high number.")

WITH imp_h2 (leaky):    accuracy = 1.000  <- looks amazing, and it should worry you
WITHOUT imp_h2 (honest): accuracy = 0.715  <- this is the real number

imp_h2 is in the label formula itself (declining_h2 = imp_h2 < 0.8*imp_h1),
so handing it to the model as a feature is handing it the answer. Deleting it and keeping
the honest score is the actual deliverable here — not the high number.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation: this contract only holds for one mid-panel month, on an unbalanced panel, and it cannot tell you anything about clients with thin history.**

Specifically:
- `dim_clients.gsc_data_start` varies a lot per client (per the flyrank-data skill's own warning) — some clients don't have 15 clean days of pre-March history at all, and those rows get silently dropped by the `imp_h1 >= 100` availability filter rather than showing up as an honest zero. The row count in 3d is smaller than the full client roster because of this, not because those clients don't exist.
- This is **one month, once**. A within-month h1-vs-h2 comparison can't distinguish a real decline from ordinary week-to-week noise or a one-off event (a holiday, a site incident) that happened to land in the second half of March specifically. Nothing here generalizes to a different month without re-running the same checks.
- `ga4_data_available IS TRUE` (checked in 3c) matters because rows before a client's GA4 start date are zero-filled, not missing — a naive `fillna(0)` or an unfiltered join would silently treat 'no GA4 yet' as 'zero engagement,' which is a different claim.
- This whole notebook intentionally avoids the sealed final month (`2026-06`) per the assignment's warning — which means I have not yet checked whether these same patterns hold outside March, and I won't know until the capstone's held-out evaluation.

In [9]:
# Quantify the first limitation named above: how much of the client roster actually
# has enough pre-March history to be eligible for this month's h1/h2 comparison at all?
coverage = con.sql(f"""
    SELECT
        COUNT(*) AS total_clients,
        SUM(CASE WHEN gsc_data_start <= DATE '{MONTH_START}' THEN 1 ELSE 0 END) AS eligible_clients
    FROM {TABLES['dim_clients']}
""").df()
coverage["eligible_pct"] = (coverage["eligible_clients"] / coverage["total_clients"] * 100).round(1)
coverage

,total_clients,eligible_clients,eligible_pct
0,104,52.0,50.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.